In [0]:
# ============================================================
# STEP 1 - verify access to REAL shared data + a place to write results.
# Success = a row count prints and "write OK: True" prints, with no red error.
# ============================================================

# 1a. READ a real shared table from the 'samples' share (read-only, no guesswork).
src = spark.table("samples.bakehouse.sales_transactions")
print("sales_transactions rows:", src.count())   # a real count of real rows
src.printSchema()                                 # the real columns we'll monitor

# 1b. Point our monitor's OUTPUT at a catalog we can write to (yours is 'workspace').
CATALOG = "workspace"
SCHEMA  = "sla_monitor"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# 1c. Prove the write path works before we build anything on it.
spark.sql(f"CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}._write_test AS SELECT 1 AS ok")
print("write OK:", spark.table(f"{CATALOG}.{SCHEMA}._write_test").collect()[0]["ok"] == 1)

DataFrame[]

In [0]:
# ============================================================
# STEP 2 - profile the REAL table. Measure only; no pass/fail yet.
# We look at completeness (nulls), uniqueness (dup keys), and validity (bad values).
# ============================================================
from pyspark.sql import functions as F

TABLE = "samples.bakehouse.sales_transactions"
df = spark.table(TABLE)

total = df.count()
print("total rows:", total)

# --- completeness: null counts on the columns a sale can't be missing ----------
null_counts = df.select([
    F.count(F.when(F.col(c).isNull(), 1)).alias(c)
    for c in ["transactionID", "customerID", "franchiseID", "dateTime",
              "product", "quantity", "unitPrice", "totalPrice", "paymentMethod"]
])
print("\nnull counts per column:")
null_counts.show(truncate=False)

# --- uniqueness: is transactionID actually a unique key? -----------------------
distinct_ids = df.select("transactionID").distinct().count()
print("duplicate transactionIDs:", total - distinct_ids)

# --- validity: values that make no business sense -------------------------------
bad_qty   = df.filter(F.col("quantity")   <= 0).count()   # a sale of 0 or fewer items
bad_price = df.filter(F.col("unitPrice")  <= 0).count()   # non-positive unit price
# does totalPrice reconcile with quantity * unitPrice? (allow tiny rounding slack)
mismatch  = df.filter(F.abs(F.col("totalPrice") - F.col("quantity") * F.col("unitPrice")) > 0.01).count()
print("\ninvalid quantity (<=0):", bad_qty)
print("invalid unitPrice (<=0):", bad_price)
print("totalPrice != quantity*unitPrice:", mismatch)

# --- recency: newest transaction (informational — this share is static) ---------
print("\nnewest dateTime:", df.agg(F.max("dateTime")).collect()[0][0])

total rows: 3333

null counts per column:
+-------------+----------+-----------+--------+-------+--------+---------+----------+-------------+
|transactionID|customerID|franchiseID|dateTime|product|quantity|unitPrice|totalPrice|paymentMethod|
+-------------+----------+-----------+--------+-------+--------+---------+----------+-------------+
|0            |0         |0          |0       |0      |0       |0        |0         |0            |
+-------------+----------+-----------+--------+-------+--------+---------+----------+-------------+

duplicate transactionIDs: 0

invalid quantity (<=0): 0
invalid unitPrice (<=0): 0
totalPrice != quantity*unitPrice: 0

newest dateTime: 2024-05-17 11:48:38.084275


In [0]:
# ============================================================
# STEP 3 - declare the SLA contract as DATA (a table in your workspace).
# Each row is one rule: a check, the column it applies to, and the limit.
# Thresholds come straight from what step 2 measured on the real data.
# ============================================================
CATALOG, SCHEMA = "workspace", "sla_monitor"
TABLE = "samples.bakehouse.sales_transactions"

# (rule_id, dimension, check_type, target_column, threshold, severity, description)
# threshold semantics: for *_max rules it's the max allowed FRACTION of bad rows (0.0 = none).
rules = [
    ("min_rows",      "completeness", "row_count_min",   None,           3000,  "SEV2", "at least 3000 rows present"),
    ("null_txn",      "completeness", "null_rate_max",   "transactionID", 0.0,   "SEV1", "transactionID never null"),
    ("null_customer", "completeness", "null_rate_max",   "customerID",    0.0,   "SEV2", "customerID never null"),
    ("null_total",    "completeness", "null_rate_max",   "totalPrice",    0.0,   "SEV2", "totalPrice never null"),
    ("dup_txn",       "uniqueness",   "dup_rate_max",    "transactionID", 0.0,   "SEV1", "transactionID must be unique"),
    ("bad_qty",       "validity",     "nonpositive_max", "quantity",      0.0,   "SEV2", "quantity must be > 0"),
    ("bad_price",     "validity",     "nonpositive_max", "unitPrice",     0.0,   "SEV2", "unitPrice must be > 0"),
    ("total_recon",   "validity",     "reconcile_max",   "totalPrice",    0.0,   "SEV1", "totalPrice = quantity * unitPrice"),
]

contract = spark.createDataFrame(
    [(TABLE,) + r for r in rules],
    "table_name string, rule_id string, dimension string, check_type string, "
    "target_column string, threshold double, severity string, description string"
)
contract.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.sla_contract")

print("contract rules saved:", contract.count())
contract.select("rule_id","dimension","check_type","target_column","threshold","severity").show(truncate=False)

contract rules saved: 8
+-------------+------------+---------------+-------------+---------+--------+
|rule_id      |dimension   |check_type     |target_column|threshold|severity|
+-------------+------------+---------------+-------------+---------+--------+
|min_rows     |completeness|row_count_min  |NULL         |3000.0   |SEV2    |
|null_txn     |completeness|null_rate_max  |transactionID|0.0      |SEV1    |
|null_customer|completeness|null_rate_max  |customerID   |0.0      |SEV2    |
|null_total   |completeness|null_rate_max  |totalPrice   |0.0      |SEV2    |
|dup_txn      |uniqueness  |dup_rate_max   |transactionID|0.0      |SEV1    |
|bad_qty      |validity    |nonpositive_max|quantity     |0.0      |SEV2    |
|bad_price    |validity    |nonpositive_max|unitPrice    |0.0      |SEV2    |
|total_recon  |validity    |reconcile_max  |totalPrice   |0.0      |SEV1    |
+-------------+------------+---------------+-------------+---------+--------+



In [0]:
# ============================================================
# STEP 4 - evaluate every contract rule against the REAL table.
# Reads workspace.sla_monitor.sla_contract, runs each check, returns pass/fail.
# One dispatch function per check_type keeps the engine small and extensible.
# ============================================================
from pyspark.sql import functions as F
from datetime import datetime, timezone

CATALOG, SCHEMA = "workspace", "sla_monitor"
TABLE = "samples.bakehouse.sales_transactions"

df = spark.table(TABLE)
total = df.count()                      # measured once, reused by the rate checks
contract = spark.table(f"{CATALOG}.{SCHEMA}.sla_contract").collect()

# Each check returns (observed_value, breached?). 'observed' is the actual measured
# number so the result is explainable, not just a boolean.
def evaluate(rule):
    ct, col, thr = rule["check_type"], rule["target_column"], rule["threshold"]

    if ct == "row_count_min":                         # enough rows present?
        observed = total
        return observed, observed < thr

    if ct == "null_rate_max":                         # fraction of NULLs in a column
        bad = df.filter(F.col(col).isNull()).count()
        observed = bad / total
        return observed, observed > thr

    if ct == "dup_rate_max":                          # fraction of rows that are dup keys
        distinct = df.select(col).distinct().count()
        observed = (total - distinct) / total
        return observed, observed > thr

    if ct == "nonpositive_max":                       # fraction of rows with value <= 0
        bad = df.filter(F.col(col) <= 0).count()
        observed = bad / total
        return observed, observed > thr

    if ct == "reconcile_max":                         # fraction where totalPrice != qty*unitPrice
        bad = df.filter(F.abs(F.col("totalPrice") - F.col("quantity") * F.col("unitPrice")) > 0.01).count()
        observed = bad / total
        return observed, observed > thr

    return None, False                                # unknown check type -> treated as pass

# run every rule and collect a result row per rule
run_ts = datetime.now(timezone.utc)
results = []
for r in contract:
    observed, breached = evaluate(r)
    results.append((
        run_ts, r["table_name"], r["rule_id"], r["dimension"], r["check_type"],
        r["target_column"], float(r["threshold"]),
        float(observed) if observed is not None else None,
        "BREACH" if breached else "OK",
        r["severity"] if breached else None,     # severity only matters when breached
        r["description"],
    ))

results_df = spark.createDataFrame(results,
    "run_ts timestamp, table_name string, rule_id string, dimension string, check_type string, "
    "target_column string, threshold double, observed double, status string, "
    "breach_severity string, description string")

print("evaluated rules:", results_df.count())
results_df.select("rule_id","dimension","threshold","observed","status","breach_severity").show(truncate=False)

evaluated rules: 8
+-------------+------------+---------+--------+------+---------------+
|rule_id      |dimension   |threshold|observed|status|breach_severity|
+-------------+------------+---------+--------+------+---------------+
|min_rows     |completeness|3000.0   |3333.0  |OK    |NULL           |
|null_txn     |completeness|0.0      |0.0     |OK    |NULL           |
|null_customer|completeness|0.0      |0.0     |OK    |NULL           |
|null_total   |completeness|0.0      |0.0     |OK    |NULL           |
|dup_txn      |uniqueness  |0.0      |0.0     |OK    |NULL           |
|bad_qty      |validity    |0.0      |0.0     |OK    |NULL           |
|bad_price    |validity    |0.0      |0.0     |OK    |NULL           |
|total_recon  |validity    |0.0      |0.0     |OK    |NULL           |
+-------------+------------+---------+--------+------+---------------+



In [0]:
# ============================================================
# STEP 5a - append this evaluation to a history table.
# Append-only (not overwrite) so we accumulate runs over time -> trends, MTTR, streaks.
# ============================================================
(results_df.write
    .mode("append")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.sla_results"))

hist = spark.table(f"{CATALOG}.{SCHEMA}.sla_results")
print("total result rows in history:", hist.count())   # 8 after the first run
print("distinct runs so far:", hist.select("run_ts").distinct().count())

total result rows in history: 8
distinct runs so far: 1


In [0]:
# ============================================================
# STEP 5b - PROOF: build a deliberately corrupted COPY, re-evaluate, watch it fail.
# We never touch the shared source; we make a broken copy in OUR schema and point
# the same engine at it. This is the "does the monitor actually catch problems?" test.
# ============================================================
from pyspark.sql import functions as F

BROKEN = f"{CATALOG}.{SCHEMA}.sales_transactions_broken"

base = spark.table(TABLE)
# inject three real defects:
#  1) NULL out transactionID on ~1% of rows        -> null_txn (SEV1) should BREACH
#  2) flip some quantities negative                 -> bad_qty (SEV2) should BREACH,
#                                                       and totalPrice no longer reconciles -> total_recon (SEV1)
broken = (base
    .withColumn("transactionID",
        F.when(F.rand(seed=1) < 0.01, None).otherwise(F.col("transactionID")))
    .withColumn("quantity",
        F.when(F.rand(seed=2) < 0.02, -F.col("quantity")).otherwise(F.col("quantity"))))
broken.write.mode("overwrite").saveAsTable(BROKEN)

# re-run the SAME engine against the broken copy (contract unchanged)
dfb = spark.table(BROKEN)
totalb = dfb.count()
def evaluate_on(dfx, totalx, rule):
    ct, col, thr = rule["check_type"], rule["target_column"], rule["threshold"]
    if ct == "row_count_min":  o = totalx;                                                         return o, o < thr
    if ct == "null_rate_max":  o = dfx.filter(F.col(col).isNull()).count()/totalx;                 return o, o > thr
    if ct == "dup_rate_max":   o = (totalx - dfx.select(col).distinct().count())/totalx;           return o, o > thr
    if ct == "nonpositive_max":o = dfx.filter(F.col(col) <= 0).count()/totalx;                     return o, o > thr
    if ct == "reconcile_max":  o = dfx.filter(F.abs(F.col("totalPrice")-F.col("quantity")*F.col("unitPrice"))>0.01).count()/totalx; return o, o > thr
    return None, False

print("results against the BROKEN copy:")
for r in contract:
    o, breached = evaluate_on(dfb, totalb, r)
    tag = f"BREACH [{r['severity']}]" if breached else "OK"
    print(f"  {r['rule_id']:14s} observed={o:.4f}  -> {tag}")

results against the BROKEN copy:
  min_rows       observed=3333.0000  -> OK
  null_txn       observed=0.0078  -> BREACH [SEV1]
  null_customer  observed=0.0000  -> OK
  null_total     observed=0.0000  -> OK
  dup_txn        observed=0.0075  -> BREACH [SEV1]
  bad_qty        observed=0.0180  -> BREACH [SEV2]
  bad_price      observed=0.0000  -> OK
  total_recon    observed=0.0180  -> BREACH [SEV1]


In [0]:
# ============================================================
# STEP 6 - the queries a dashboard is built from, over workspace.sla_monitor.sla_results.
# Run here to confirm they work; then recreate each as a tile in a Databricks dashboard.
# ============================================================
CATALOG, SCHEMA = "workspace", "sla_monitor"
R = f"{CATALOG}.{SCHEMA}.sla_results"

# TILE 1 - latest run: current status of every rule (the on-call board)
print("=== current status (latest run) ===")
spark.sql(f"""
    WITH latest AS (SELECT max(run_ts) AS t FROM {R})
    SELECT rule_id, dimension, threshold, observed, status, breach_severity
    FROM {R} WHERE run_ts = (SELECT t FROM latest)
    ORDER BY (status='BREACH') DESC, breach_severity, rule_id
""").show(truncate=False)

# TILE 2 - headline counts for the latest run (the summary tiles)
print("=== summary (latest run) ===")
spark.sql(f"""
    WITH latest AS (SELECT max(run_ts) AS t FROM {R})
    SELECT count(*) AS rules_checked,
           sum(CASE WHEN status='BREACH' THEN 1 ELSE 0 END) AS breaches,
           sum(CASE WHEN breach_severity='SEV1' THEN 1 ELSE 0 END) AS sev1,
           round(100.0*avg(CASE WHEN status='OK' THEN 1 ELSE 0 END),1) AS pass_rate_pct
    FROM {R} WHERE run_ts = (SELECT t FROM latest)
""").show(truncate=False)

# TILE 3 - pass-rate trend across runs (fills in as you accumulate runs over time)
print("=== pass-rate by run (trend) ===")
spark.sql(f"""
    SELECT run_ts,
           round(100.0*avg(CASE WHEN status='OK' THEN 1 ELSE 0 END),1) AS pass_rate_pct,
           sum(CASE WHEN status='BREACH' THEN 1 ELSE 0 END) AS breaches
    FROM {R} GROUP BY run_ts ORDER BY run_ts
""").show(truncate=False)

=== current status (latest run) ===
+-------------+------------+---------+--------+------+---------------+
|rule_id      |dimension   |threshold|observed|status|breach_severity|
+-------------+------------+---------+--------+------+---------------+
|bad_price    |validity    |0.0      |0.0     |OK    |NULL           |
|bad_qty      |validity    |0.0      |0.0     |OK    |NULL           |
|dup_txn      |uniqueness  |0.0      |0.0     |OK    |NULL           |
|min_rows     |completeness|3000.0   |3333.0  |OK    |NULL           |
|null_customer|completeness|0.0      |0.0     |OK    |NULL           |
|null_total   |completeness|0.0      |0.0     |OK    |NULL           |
|null_txn     |completeness|0.0      |0.0     |OK    |NULL           |
|total_recon  |validity    |0.0      |0.0     |OK    |NULL           |
+-------------+------------+---------+--------+------+---------------+

=== summary (latest run) ===
+-------------+--------+----+-------------+
|rules_checked|breaches|sev1|pass_rate

In [0]:
# OPTIONAL - persist the broken evaluation as a second run so the dashboard shows
# both a healthy and a breaching run (better for a demo screenshot / trend line).
from datetime import datetime, timezone
run_ts_b = datetime.now(timezone.utc)
broken_results = []
for r in contract:
    o, breached = evaluate_on(dfb, totalb, r)
    broken_results.append((
        run_ts_b, BROKEN, r["rule_id"], r["dimension"], r["check_type"], r["target_column"],
        float(r["threshold"]), float(o) if o is not None else None,
        "BREACH" if breached else "OK", r["severity"] if breached else None, r["description"]))
spark.createDataFrame(broken_results, results_df.schema) \
     .write.mode("append").saveAsTable(f"{CATALOG}.{SCHEMA}.sla_results")
print("history now has", spark.table(f"{CATALOG}.{SCHEMA}.sla_results").select("run_ts").distinct().count(), "runs")

history now has 2 runs
